# Furniture-Aware Training (Kaggle T4, 무료)

**목표:** 빌트인 가구 환경에서 정확한 부위/가구 분류

**클래스 (10개):**
- 부위: wall, ceiling, floor, window, door
- 빌트인 가구: cabinet, kitchen_appliance, countertop_sink, kitchen_island, shelf

**전략 (Kaggle T4 12h 한도 안전):**
- 모델: **yolov11l** (47M, 큰 모델)
- imgsz: 960 (T4 안전)
- batch: 8 (T4 16GB)
- Stage 1: 80ep + Stage 2: 20ep = 100ep
- 예상 시간: 9~10시간

**Kaggle Datasets 준비:**
1. `furniture_aware.zip` 업로드 (예상 3~5GB)
2. 노트북 우측 + Add Input → 본인 dataset 추가

**런타임:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu numpy pillow

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★★★ 본인이 업로드한 Kaggle Dataset slug ★★★
# 우측 사이드바 'Input' 섹션에서 폴더 이름 확인
INPUT_DIR = Path('/kaggle/input/furniture-aware')  # 본인 dataset slug 확인
ZIP_NAME = 'furniture_aware.zip'

WORK = Path('/kaggle/working/furniture_aware')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = INPUT_DIR / ZIP_NAME
print(f'zip exists? {zip_path.exists()} -> {zip_path}')
if not zip_path.exists():
    # zip 풀려있는 경우 직접 사용
    print('zip 못 찾음. 데이터셋이 자동 압축 해제됐나 확인:')
    if INPUT_DIR.exists():
        for f in os.listdir(INPUT_DIR):
            print(f'  {f}')
assert zip_path.exists() or (INPUT_DIR / 'images').exists(), 'dataset 못 찾음. INPUT_DIR 확인'

ds_dir = WORK / 'furniture_aware'
if zip_path.exists():
    if not (ds_dir / 'images' / 'val').exists():
        if ds_dir.exists():
            shutil.rmtree(ds_dir, ignore_errors=True)
        print('Extracting...')
        with zipfile.ZipFile(zip_path) as z:
            for m in z.namelist():
                rel = m.replace('\\', '/')
                target = WORK / rel
                if rel.endswith('/'):
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(m) as src, open(target, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
else:
    # Kaggle이 자동 풀어준 경우
    ds_dir = INPUT_DIR

for s in ['train', 'val', 'test']:
    p = ds_dir / 'images' / s
    if p.exists():
        print(f'  {s}: {len(list(p.glob("*")))} images')

In [ ]:
yaml_text = f'''path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 10
names:
  0: wall
  1: ceiling
  2: floor
  3: window
  4: door
  5: cabinet_builtin
  6: kitchen_appliance
  7: countertop_sink
  8: kitchen_island
  9: shelf
'''
data_yaml = WORK / 'furniture_aware.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## Stage 1: yolov11l + 960 + batch=8 + 80ep

In [ ]:
from ultralytics import YOLO

PROJECT = Path('/kaggle/working/runs')
PROJECT.mkdir(parents=True, exist_ok=True)

print('=== Stage 1 (yolov11l + 960 + batch=8 + 80ep) ===')
model_s1 = YOLO('yolo11l.pt')
model_s1.train(
    data=str(data_yaml),
    epochs=80,
    batch=8,
    imgsz=960,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    cos_lr=True,
    patience=20,
    warmup_epochs=3,
    close_mosaic=20,
    freeze=0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5,
    shear=2.0, perspective=0.001,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.15, copy_paste=0.5,
    erasing=0.0,
    multi_scale=0.2,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage1',
    exist_ok=True
)
print('Stage 1 done.')

## Stage 2: fine-tune (lr=1e-5, freeze=10, 20ep)

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
print('=== Stage 2 ===')
model_s2 = YOLO(str(stage1_best))
model_s2.train(
    data=str(data_yaml),
    epochs=20,
    batch=8,
    imgsz=960,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-5,
    lrf=0.01,
    cos_lr=True,
    patience=10,
    warmup_epochs=1,
    close_mosaic=15,
    freeze=10,
    mosaic=0.5, mixup=0.0, copy_paste=0.2,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage2',
    exist_ok=True
)
print('Stage 2 done.')

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
stage2_best = PROJECT / 'stage2' / 'weights' / 'best.pt'
best_path = stage2_best if stage2_best.exists() else stage1_best
print(f'best.pt: {best_path}')
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')

metrics = best_model.val(data=str(data_yaml), imgsz=960, batch=8)
print(f'\n=== Furniture-Aware 최종 ===')
print(f'  mAP50:    {metrics.box.map50:.4f}')
print(f'  mAP50-95: {metrics.box.map:.4f}')
print(f'  precision: {metrics.box.mp:.4f}')
print(f'  recall:    {metrics.box.mr:.4f}')
print(f'  0.9? {"YES" if metrics.box.map50 >= 0.9 else "NO"}')

for i, n in enumerate(['wall','ceiling','floor','window','door','cabinet','appliance','counter','island','shelf']):
    if i < len(metrics.box.maps):
        print(f'  {n}: mAP50-95={metrics.box.maps[i]:.4f}')

In [ ]:
OUT = Path('/kaggle/working/furniture_aware_results')
OUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(onnx_path, OUT / 'furniture_aware.onnx')
shutil.copy2(best_path, OUT / 'furniture_aware_best.pt')
for s in ['stage1', 'stage2']:
    rcsv = PROJECT / s / 'results.csv'
    if rcsv.exists():
        shutil.copy2(rcsv, OUT / f'{s}_results.csv')
    for plot in (PROJECT / s).glob('*.png'):
        shutil.copy2(plot, OUT / f'{s}_{plot.name}')

print('Saved to:', OUT)
for f in OUT.iterdir():
    print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f}MB)')
print('\n노트북 우측 Output 탭에서 다운로드 가능')